# custom-sam-peft — GPU Test Runner

Per-test cells for GPU-gated tests. Press play on the cell for the test you want;
the cleanup trailer runs after every test (pass or fail), so the next test starts
from a clean slate without a full runtime restart.

**Prereqs:**
1. Runtime → Change runtime type → **T4 GPU** (or better).
2. In Colab Secrets (left sidebar 🔑), add:
   - `HF_TOKEN` — Hugging Face access token with read access to gated `facebook/sam3.1`.
   - `GH_TOKEN` — GitHub fine-grained personal access token with **Contents: Read** on this repo. **Required only if the repo is private**; can be omitted for public repos.
3. **Set `BRANCH` in the next cell** (default is `"main"`).
4. Run setup cells (collapsed below) once per session.
5. Press play on each test cell in turn. Cells are the canonical merge gate — every cell green = PR ready.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SET THE BRANCH HERE before running anything else.
#   - default: "main"
#   - for a PR/feature branch, e.g.: BRANCH = "manual-gpu-pass-44"
# ════════════════════════════════════════════════════════════════════════
BRANCH = "main"

REPO = "NguyenJus/custom-sam-peft"
print(f"REPO   = {REPO}")
print(f"BRANCH = {BRANCH}")


### GPU guard (runs first)

This first code cell asserts a CUDA GPU is available before any install
or test step. Colab can pin the **accelerator class** (GPU vs. CPU) via
notebook metadata (`metadata.colab.accelerator: "GPU"`), but it cannot
pin the **GPU model** — the most common assignment is a T4, but Colab
may serve a V100, A100, L4, or others depending on availability. The
test suite is calibrated for a T4; the guard warns (does not fail) if a
different GPU is detected.


In [ ]:
# GPU guard. Runs FIRST so a misconfigured runtime fails loudly before any
# slow install or test step. Two assertions:
#   1. nvidia-smi must succeed AND report at least one GPU. If it doesn't,
#      the runtime is CPU-only — change Runtime → Change runtime type → GPU.
#   2. If the GPU is not a T4, print a WARN: the test suite's timing /
#      memory assumptions are calibrated for a Colab T4, and other GPUs
#      (V100, A100, L4, …) may show different characteristics. The cell does
#      NOT fail in that case — other GPUs are usually fine, just unverified.
import subprocess
import sys

try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        check=True,
        capture_output=True,
        text=True,
    )
except FileNotFoundError as e:
    raise RuntimeError(
        "nvidia-smi not found; this runtime is CPU-only. "
        "Runtime → Change runtime type → GPU (T4 recommended) → Save."
    ) from e
except subprocess.CalledProcessError as e:
    raise RuntimeError(
        f"nvidia-smi failed (returncode={e.returncode}). "
        f"stderr: {e.stderr.strip()}"
    ) from e

gpus = [line.strip() for line in out.stdout.splitlines() if line.strip()]
if not gpus:
    raise RuntimeError(
        "nvidia-smi reported no GPUs. "
        "Runtime → Change runtime type → GPU (T4 recommended) → Save."
    )

print(f"GPU(s) detected: {gpus}")
if not any("T4" in g for g in gpus):
    print(
        f"WARN: detected GPU is not a T4 ({gpus}). The test suite's timing "
        "and memory assumptions are calibrated for a T4; other GPUs are "
        "usually fine but unverified.",
        file=sys.stderr,
    )


In [ ]:
# Cell 1: Clone + checkout. Done BEFORE any torch/numpy import so the pip
# install in Cell 2 can downgrade numpy without Colab forcing a runtime restart.
import os
import subprocess

try:
    from google.colab import userdata  # type: ignore

    gh_token = userdata.get("GH_TOKEN")
except Exception:
    gh_token = None

clone_url = (
    f"https://{gh_token}@github.com/{REPO}.git" if gh_token else f"https://github.com/{REPO}.git"
)

if not os.path.isdir("custom-sam-peft"):
    subprocess.run(["git", "clone", clone_url], check=True)
subprocess.run(["git", "-C", "custom-sam-peft", "fetch", "--all"], check=True)
subprocess.run(["git", "-C", "custom-sam-peft", "checkout", BRANCH], check=True)
os.chdir("custom-sam-peft")
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

In [ ]:
# Install runtime + GPU + test deps + pytest-sugar (per-test progress bar).
#
# numpy/scipy/transformers pins must stay on the SAME pip install line so
# pip's resolver honors them in one pass; splitting them re-resolves and
# tries to upgrade numpy past sam3's <2 cap, cascading scipy backtracks.
# torchao>=0.16.0 is pinned because peft 0.19+ raises ImportError on the
# torchao 0.10.0 that Colab preinstalls.
%pip install -e ".[qlora,dev,tensorboard]" \
    "numpy==1.26.4" "scipy==1.13.1" "transformers==5.0.0" \
    "huggingface_hub>=1.15" \
    "torchao>=0.16.0" \
    "pytest-sugar"
!python -c "import custom_sam_peft; print('custom_sam_peft OK:', custom_sam_peft.__file__)"

# Fail-fast check: if Colab preloaded numpy 2.x into the kernel before our
# install ran, the on-disk numpy is now 1.26.4 but sys.modules still holds
# the old 2.x. Auto-restart attempts proved unreliable; instead, raise a
# clear instruction.
import sys

if "numpy" in sys.modules:
    import importlib.metadata
    _in_mem = sys.modules["numpy"].__version__
    _on_disk = importlib.metadata.version("numpy")
    if _in_mem != _on_disk:
        raise RuntimeError(
            f"numpy mismatch: kernel has {_in_mem}, disk has {_on_disk}.\n"
            "Colab pre-imported the old version before our pin took effect.\n"
            "Runtime → Restart session, then Run all (the install re-runs as a no-op)."
        )


In [ ]:
# Cell 4: HF auth (token in Colab Secrets).
import os

from google.colab import userdata

token = userdata.get("HF_TOKEN")
assert token, "HF_TOKEN missing from Colab Secrets. See the prereqs cell."
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token  # huggingface-cli reads this name too

In [ ]:
# Cell 5: Download the SAM 3.1 checkpoint (gated; HF_TOKEN required).
# `huggingface-cli` is deprecated; the new CLI is `hf` (huggingface_hub >= 1.13).
!hf download facebook/sam3.1 --local-dir models/sam3.1

## Test cells — press play for one test

Each cell pulls the latest branch tip, runs one test (or the inspection tier as
one batch), and runs the GPU cleanup trailer regardless of pass/fail. The
inspection tier is cheap (no training) and a good first check after each push;
the training tiers are expensive and best run individually so a mid-run failure
doesn't take 40 minutes of others down with it.

The long-running training cells stream the trainer's WARNING-level logs live
(`-s --log-cli-level=WARNING`), so you can see NaN-class skips or training
events as they happen instead of waiting for the test to finish.


In [ ]:
%%bash
# Inspection tier — 9 forward-only tests. ~5 min wall on T4.
# Validates load-time / config / structural contracts; cheapest gate.
# pytest-sugar provides the per-test progress bar.
cleanup() {
    echo ""
    echo "=== GPU CLEANUP ==="
    ORPHANS=$(nvidia-smi --query-compute-apps=pid --format=csv,noheader | tr -d ' ')
    if [ -n "$ORPHANS" ]; then
        echo "Killing GPU processes still holding memory: $ORPHANS"
        echo "$ORPHANS" | xargs -r kill -9 2>/dev/null
        sleep 1
    fi
    USED=$(nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits | tr -d ' ')
    [ "$USED" -lt 100 ] && echo "GPU memory clean (${USED} MiB used)" || echo "WARN: GPU memory not released (${USED} MiB used)"
    echo ""
    echo "Test exit code: ${TEST_EXIT:-<interrupted>} (0 = pass)"
}
trap cleanup EXIT
set -e
git pull --ff-only
set +e
python -m pytest tests/integration/ \
    -v --tb=long --no-cov -m 'gpu_inspection'
TEST_EXIT=$?


In [ ]:
%%bash
# QLoRA fast smoke — 3 training steps, no final eval. ~3 min wall on T4.
# Iteration aid; same code path as the full overfit but with weaker
# assertions (finite-value only). Run after each push during dtype/grad
# debug cycles. -s --log-cli-level=WARNING streams trainer warnings live.
cleanup() {
    echo ""
    echo "=== GPU CLEANUP ==="
    ORPHANS=$(nvidia-smi --query-compute-apps=pid --format=csv,noheader | tr -d ' ')
    if [ -n "$ORPHANS" ]; then
        echo "Killing GPU processes still holding memory: $ORPHANS"
        echo "$ORPHANS" | xargs -r kill -9 2>/dev/null
        sleep 1
    fi
    USED=$(nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits | tr -d ' ')
    [ "$USED" -lt 100 ] && echo "GPU memory clean (${USED} MiB used)" || echo "WARN: GPU memory not released (${USED} MiB used)"
    echo ""
    echo "Test exit code: ${TEST_EXIT:-<interrupted>} (0 = pass)"
}
trap cleanup EXIT
set -e
git pull --ff-only
set +e
python -m pytest tests/gpu/test_real_train_qlora.py::test_qlora_smoke_fast \
    -v -s --log-cli-level=WARNING --tb=long --no-cov -m 'gpu'
TEST_EXIT=$?


In [ ]:
%%bash
# LoRA release-tier overfit — 50 training steps + final Evaluator.
# ~18 min wall on T4. Asserts loss drop ≤ 0.70 of start, peak VRAM ≤ 14 GB.
# -s --log-cli-level=WARNING streams trainer warnings live (NaN-class skips, etc).
cleanup() {
    echo ""
    echo "=== GPU CLEANUP ==="
    ORPHANS=$(nvidia-smi --query-compute-apps=pid --format=csv,noheader | tr -d ' ')
    if [ -n "$ORPHANS" ]; then
        echo "Killing GPU processes still holding memory: $ORPHANS"
        echo "$ORPHANS" | xargs -r kill -9 2>/dev/null
        sleep 1
    fi
    USED=$(nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits | tr -d ' ')
    [ "$USED" -lt 100 ] && echo "GPU memory clean (${USED} MiB used)" || echo "WARN: GPU memory not released (${USED} MiB used)"
    echo ""
    echo "Test exit code: ${TEST_EXIT:-<interrupted>} (0 = pass)"
}
trap cleanup EXIT
set -e
git pull --ff-only
set +e
python -m pytest tests/gpu/test_real_train_overfits.py \
    -v -s --log-cli-level=WARNING --tb=long --no-cov -m 'gpu'
TEST_EXIT=$?


In [ ]:
%%bash
# QLoRA release-tier overfit — 50 training steps + final Evaluator.
# ~18 min wall on T4. Asserts loss drop ≤ 0.75 of start, peak VRAM ≤ 10 GB.
# -s --log-cli-level=WARNING streams trainer warnings live.
cleanup() {
    echo ""
    echo "=== GPU CLEANUP ==="
    ORPHANS=$(nvidia-smi --query-compute-apps=pid --format=csv,noheader | tr -d ' ')
    if [ -n "$ORPHANS" ]; then
        echo "Killing GPU processes still holding memory: $ORPHANS"
        echo "$ORPHANS" | xargs -r kill -9 2>/dev/null
        sleep 1
    fi
    USED=$(nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits | tr -d ' ')
    [ "$USED" -lt 100 ] && echo "GPU memory clean (${USED} MiB used)" || echo "WARN: GPU memory not released (${USED} MiB used)"
    echo ""
    echo "Test exit code: ${TEST_EXIT:-<interrupted>} (0 = pass)"
}
trap cleanup EXIT
set -e
git pull --ff-only
set +e
python -m pytest tests/gpu/test_real_train_qlora.py::test_qlora_overfits_in_50_steps \
    -v -s --log-cli-level=WARNING --tb=long --no-cov -m 'gpu'
TEST_EXIT=$?


In [ ]:
%%bash
# End-to-end CLI smoke — exercises `custom-sam-peft run` plus disk-artifact assertions
# (run dir layout, adapter save, metrics.json). ~18 min wall on T4.
# -s --log-cli-level=WARNING streams trainer warnings live.
cleanup() {
    echo ""
    echo "=== GPU CLEANUP ==="
    ORPHANS=$(nvidia-smi --query-compute-apps=pid --format=csv,noheader | tr -d ' ')
    if [ -n "$ORPHANS" ]; then
        echo "Killing GPU processes still holding memory: $ORPHANS"
        echo "$ORPHANS" | xargs -r kill -9 2>/dev/null
        sleep 1
    fi
    USED=$(nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits | tr -d ' ')
    [ "$USED" -lt 100 ] && echo "GPU memory clean (${USED} MiB used)" || echo "WARN: GPU memory not released (${USED} MiB used)"
    echo ""
    echo "Test exit code: ${TEST_EXIT:-<interrupted>} (0 = pass)"
}
trap cleanup EXIT
set -e
git pull --ff-only
set +e
python -m pytest tests/gpu/test_run_end_to_end_gpu.py \
    -v -s --log-cli-level=WARNING --tb=long --no-cov -m 'gpu'
TEST_EXIT=$?


## Reading test results

Scroll to the bottom of any test cell's output and read pytest's final summary
line (e.g. `========= 4 passed in 87.3s =========`). That line is the
pass/fail signal. The cleanup trailer below it confirms GPU state for the
next cell.

If a cell fails:
1. The cleanup trailer still ran — GPU memory is reclaimed; you don't need
   to restart the runtime.
2. Read the traceback above the summary line.
3. Push a fix locally, re-press the same cell. The `git pull --ff-only` at
   the top picks up the new commit.
